# Práctica 7 — Análisis funcional con decoupler: actividad de TFs y rutas en bulk RNA-seq

**Módulo de Transcriptómica — Máster en Bioinformática**

El RNA-seq bulk produce miles de valores de expresión difíciles de interpretar gen a gen. Una forma de resumirlos es **inferir actividades de factores de transcripción (TFs) y de rutas** a partir de conocimiento previo, que es lo que hace `decoupler`.

En esta práctica usamos el dataset **GSE151251**: células madre hematopoyéticas (HSCs) humanas, 3 controles frente a 3 tratadas con la citoquina **TGFβ**.

**Flujo:**

1. Control de calidad y filtrado de genes (criterio `filterByExpr` de edgeR)
2. Expresión diferencial con PyDESeq2
3. **Actividad de TFs** con la red CollecTRI + método ULM
4. **Actividad de rutas** con PROGENy + método MLM
5. **Enriquecimiento** con Hallmark (MSigDB) + ORA
6. *(Opcional)* Interacciones ligando–receptor con LIANA

> **Idea clave**: a diferencia del ORA clásico, `decoupler` usa **pesos** de las interacciones (activadora/inhibidora) y el estadístico completo de todos los genes, lo que permite inferir *actividad* (activada/inhibida) y no solo "enriquecimiento".

## 1. Instalación

Instalamos `decoupler` (versión 2.x), `scanpy` (manejo de AnnData), `pydeseq2` (expresión diferencial) e `igraph` (necesario para dibujar las redes de TFs).

> Si Colab pide **reiniciar la sesión** tras la instalación, hazlo y continúa desde la celda siguiente.

In [ ]:
!pip install -q decoupler scanpy pydeseq2 igraph

import importlib.metadata
for pkg in ["decoupler", "scanpy", "pydeseq2", "anndata"]:
    print(pkg, importlib.metadata.version(pkg))

## 2. Datos

Los datos crudos (conteos de `featureCounts` sobre GSE151251) están en el repositorio del curso. Si la descarga falla, sube `counts.txt` manualmente con el panel de archivos.

In [ ]:
import urllib.request
from pathlib import Path

DATA_DIR = Path("datos_decoupler")
DATA_DIR.mkdir(exist_ok=True)

URL = "https://raw.githubusercontent.com/victorsanchezarevalo/transcriptomica-master/main/practicas/07-decoupler-bulk/datos/counts.txt"
destino = DATA_DIR / "counts.txt"

if not destino.exists():
    print("Descargando counts.txt ...")
    urllib.request.urlretrieve(URL, destino)
print(f"counts.txt: {destino.stat().st_size / 1e6:.2f} MB")

In [ ]:
import numpy as np
import pandas as pd
from anndata import AnnData
import decoupler as dc
import scanpy as sc
import matplotlib.pyplot as plt

# Leer la matriz: genes en filas (índice = GeneName), muestras en columnas
adata = pd.read_csv(DATA_DIR / "counts.txt", index_col=2, sep="\t").iloc[:, 5:].T
print(f"Matriz: {adata.shape[0]} muestras x {adata.shape[1]} genes")

# Convertir a AnnData
adata = AnnData(adata)
adata.X = adata.X.astype(np.float32)
adata.var_names_make_unique()

# Metadatos a partir de los nombres de muestra
adata.obs["condition"] = ["control" if "-Ctrl" in s else "treatment" for s in adata.obs.index]
adata.obs["sample_id"] = [s.split("_")[0] for s in adata.obs.index]
adata.obs

## 3. Control de calidad: filtrado de genes

Antes de nada eliminamos genes con muy pocas lecturas (ruido). Seguimos la estrategia de `filterByExpr` de edgeR: un gene se conserva si tiene al menos `min_count` lecturas en una proporción `min_prop` de las muestras del grupo más pequeño, y al menos `min_total_count` lecturas en total.

Primero visualizamos el efecto de los umbrales (jugad con `min_count` y `min_total_count` para ver cómo cambia):

In [ ]:
dc.pl.filter_by_expr(adata, group="condition", min_count=10, min_total_count=15, large_n=1, min_prop=1)
plt.show()

In [ ]:
# Aplicar el filtro
genes = dc.pp.filter_by_expr(
    adata, group="condition",
    min_count=10, min_total_count=15,
    large_n=1, min_prop=1,
    inplace=False,
)
adata = adata[:, genes].copy()
print(f"Genes conservados: {adata.shape[1]}")

## 4. Expresión diferencial con PyDESeq2

Comparamos **treatment (TGFβ) vs control**. Usaremos después la columna `stat` (estadístico t del test de Wald), mejor que el log2FC como medida de cambio porque incorpora la significación.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

dds = DeseqDataSet(adata=adata, design="~condition", refit_cooks=True)
dds.deseq2()

stat_res = DeseqStats(dds, contrast=["condition", "treatment", "control"])
stat_res.summary()
results_df = stat_res.results_df.copy()
print(f"Genes con FDR < 0.01: {(results_df['padj'] < 0.01).sum()}")
results_df.head()

In [ ]:
# Volcano plot
dc.pl.volcano(data=results_df, x="log2FoldChange", y="padj", top=20, figsize=(5, 5))
plt.show()

In [ ]:
# Matriz de estadísticos para decoupler: una fila = un contraste
mat = results_df[["stat"]].T.rename(index={"stat": "treatment.vs.control"})
mat

## 5. Actividad de factores de transcripción (TFs)

### Red CollecTRI

[CollecTRI](https://github.com/saezlab/CollecTRI) es una red regulatoria curada que relaciona TFs con sus genes diana, con **peso +1 (activación) o −1 (represión)**.

### Inferencia con ULM (Univariate Linear Model)

Para cada TF, ULM ajusta un modelo lineal que predice los estadísticos de expresión a partir de los pesos TF–gen. El **t-value** resultante es el *score de actividad* del TF: positivo = activado en el tratamiento, negativo = inhibido.

In [ ]:
# Descargar la red CollecTRI (necesita internet)
collectri = dc.op.collectri(organism="human", remove_complexes=False)
print(f"Red CollecTRI: {collectri.shape[0]} interacciones, {collectri['source'].nunique()} TFs")
collectri.head()

In [ ]:
# Inferir actividades de TFs con ULM
tf_acts, tf_pvals = dc.mt.ulm(data=mat, net=collectri)
tf_acts

In [ ]:
# Barplot de los TFs más activados (arriba) e inhibidos (abajo)
dc.pl.barplot(data=tf_acts, name="treatment.vs.control", top=25, vertical=True, figsize=(3, 6))
plt.show()

Podemos dibujar la **red** de los TFs más cambiantes, coloreando los nodos por actividad del TF y por expresión de sus genes diana. Si un TF está activado, sus diana activadas (peso +1) deberían aparecer sobrerrepresentadas entre los genes que suben:

In [ ]:
# Los 4 TFs con mayor |actividad|
top_tfs = tf_acts.T["treatment.vs.control"].abs().nlargest(4).index.tolist()
print("Top TFs:", top_tfs)

dc.pl.network(
    net=collectri, data=mat, score=tf_acts,
    sources=top_tfs, targets=15,
    size_node=50, vcenter=True, figsize=(5, 5),
)
plt.show()

In [ ]:
# Zoom a un TF concreto: volcano coloreando sus genes diana
dc.pl.volcano(data=results_df, x="log2FoldChange", y="padj",
              net=collectri, name="SRF", top=10, figsize=(5, 5))
plt.show()

## 6. Actividad de rutas con PROGENy

[PROGENy](https://saezlab.github.io/progeny/) contiene 14 rutas de señalización de cáncer con sus genes diana y pesos calculados a partir de miles de experimentos públicos.

Usamos el método **MLM** (Multivariate Linear Model): un único modelo multivariante que estima la actividad de **todas las rutas a la vez**, teniendo en cuenta que comparten genes diana.

In [ ]:
# Descargar los pesos de PROGENy (top 500 genes por ruta)
progeny = dc.op.progeny(organism="human", top=500)
print(f"PROGENy: {progeny.shape[0]} interacciones, {progeny['source'].nunique()} rutas")
progeny.head()

In [ ]:
# Inferir actividades de rutas con MLM
pathway_acts, pathway_pvals = dc.mt.mlm(data=mat, net=progeny)
pathway_acts.T.sort_values(by="treatment.vs.control", ascending=False)

In [ ]:
# Barplot de actividades de rutas
dc.pl.barplot(data=pathway_acts, name="treatment.vs.control", top=25, vertical=False, figsize=(6, 3))
plt.show()

> **¿Tiene sentido biológico?** Las células se trataron con la citoquina **TGFβ**: la ruta TGFb debería aparecer como la más activada. Comprobadlo en el barplot.

Podemos visualizar los genes diana de la ruta TGFb en un *scatter plot*: eje x = estadístico del gen, eje y = peso en PROGENy. La activación se manifiesta porque los genes con peso positivo tienen estadísticos positivos (1er cuadrante) y los de peso negativo, negativos (3er cuadrante):

In [ ]:
dc.pl.source_targets(data=results_df, net=progeny, x="stat", y="weight", name="TGFb", top=15)
plt.show()

## 7. Enriquecimiento con Hallmark (MSigDB)

Por último, inferimos actividades para colecciones de genes anotados a procesos biológicos generales. Usamos los **50 gene sets Hallmark** de MSigDB.

Aplicamos **ORA** sobre el vector de estadísticos completo (`dc.mt.ora`), que testea si los genes en los extremos del ranking pertenecen a cada gene set.

In [ ]:
# Descargar Hallmark
hallmark = dc.op.hallmark(organism="human")
print(f"Hallmark: {hallmark['source'].nunique()} gene sets")
hallmark.head()

In [ ]:
# Enriquecimiento ORA sobre el ranking completo de estadísticos
ora_acts, ora_pvals = dc.mt.ora(data=mat, net=hallmark)

ora_acts.T.sort_values(by="treatment.vs.control", ascending=False).head(10)

In [ ]:
# Barplot de los términos más enriquecidos
dc.pl.barplot(data=ora_acts, name="treatment.vs.control", top=20, vertical=True, figsize=(3, 6))
plt.show()

También podemos dibujar el **running score** tipo GSEA de un gene set concreto, por ejemplo `TGF_BETA_SIGNALING`:

In [ ]:
dc.pl.leading_edge(df=results_df, net=hallmark, stat="stat", name="TGF_BETA_SIGNALING")
plt.show()

## 8. *(Opcional)* Interacciones ligando–receptor con LIANA

Como extra, podemos inferir qué pares **ligando–receptor** están corregulados, usando el recurso de interacciones de [LIANA](https://liana-py.readthedocs.io/). La idea: si en el tratamiento suben (o bajan) coordinadamente un ligando y su receptor, esa interacción puede estar activa.

> Requiere instalar `liana` (pesado, ~2 min) y conexión a internet.

In [ ]:
!pip install -q liana
import liana as ln
print("liana instalado")

In [ ]:
# Recurso de interacciones ligando-receptor
lr = ln.resource.select_resource()
print(f"{lr.shape[0]} pares ligando-receptor")
lr["interaction"] = lr["ligand"] + " → " + lr["receptor"]

# Formato largo: una fila por interacción y gen (ligando o receptor), columnas source/target
df1 = lr[["interaction", "ligand"]].rename(columns={"ligand": "genes"})
df2 = lr[["interaction", "receptor"]].rename(columns={"receptor": "genes"})
lr_net = (pd.concat([df1, df2]).drop_duplicates().reset_index(drop=True)
          .rename(columns={"interaction": "source", "genes": "target"}))
print(f"{lr_net.shape[0]} registros, {lr_net['source'].nunique()} interacciones")
lr_net.head()

In [ ]:
# Inferir actividad de las interacciones con ULM
lr_score, lr_pvalue = dc.mt.ulm(data=mat, net=lr_net, tmin=2)
dc.pl.barplot(lr_score, "treatment.vs.control", top=25, vertical=True)
plt.show()

## 9. Preguntas de reflexión

1. **Validación biológica**: ¿qué ruta de PROGENy sale más activada? ¿Por qué es una buena señal de que el análisis funciona?
2. **TFs**: ¿qué TFs aparecen activados e inhibidos? Buscad 2–3 en la literatura: ¿tienen relación conocida con TGFβ o con biología de células madre?
3. **ULM vs MLM**: ¿por qué MLM (multivariante) puede ser más adecuado que ULM cuando las rutas comparten muchos genes diana?
4. **Pesos vs sin pesos**: compara el resultado de PROGENy (con pesos y dirección) con el ORA de Hallmark (sin pesos). ¿Qué información se pierde en el ORA clásico? ¿Coinciden las rutas TGFβ?
5. **Elección del estadístico**: ¿por qué usamos `stat` y no `log2FoldChange` como entrada? Prueba a repetir `dc.mt.mlm` con el log2FC y compara.
6. **Ligando–receptor** *(si hiciste la sección opcional)*: ¿qué interacciones salen más activas? ¿Encajan con el programa de transición epitelio–mesénquima (EMT) que sugiere el Hallmark `EPITHELIAL_MESENCHYMAL_TRANSITION`?

---

**Entrega**: el notebook ejecutado (o script `.py`) + un PDF con las figuras clave y las respuestas a las preguntas.